# 64x64 Rectangular-Obstacle Phase B

Attach the initial private training bundle with training, validation/calibration, and ID-pilot data only. Startup requires these three archives and their filtered manifest. It does not generate data or extract/load OOD-pilot data.

Each notebook startup allocates a unique directory under `results/64x64_rectangles_v1/runs/`. Checkpoints, histories, detector/calibration files, caches, pilot outputs, reports, and ZIPs remain inside that run directory.

After model training, detector fitting, D95 calibration, and ID-only integration pass, obtain separate approval before uploading OOD pilot. Attach it and set OOD_PILOT_RELEASE_APPROVED to True before the gated release cell. ID holdout and sealed OOD test are never included.


In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile

ON_KAGGLE = Path("/kaggle/working").is_dir()
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path.cwd()
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
PREPARED_ROOT = WORK_ROOT / "phase_b_inputs"
RUN_ID = "64x64_rectangles_v1"
INITIAL_BUNDLE_NAME = "64x64_rectangles_v1_kaggle_upload.zip"
REQUIRED_INITIAL_ARCHIVES = ("training.npz", "validation_calibration.npz", "id_pilot.npz")

def _safe_extract(archive_path, target):
    target.mkdir(parents=True, exist_ok=True)
    root = target.resolve()
    with zipfile.ZipFile(archive_path) as bundle:
        for member in bundle.infolist():
            resolved = (target / member.filename).resolve()
            if resolved != root and root not in resolved.parents:
                raise ValueError(f"Unsafe path in input bundle: {member.filename}")
        bundle.extractall(target)

def _has_initial_data(path):
    return (path / "manifest.json").is_file() and all(
        (path / name).is_file() for name in REQUIRED_INITIAL_ARCHIVES
    )

PROJECT_ROOT = None
INITIAL_EXTRACT_ROOT = None
if ON_KAGGLE:
    candidates = sorted(INPUT_ROOT.rglob(INITIAL_BUNDLE_NAME))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Attach exactly one initial Phase B bundle named {INITIAL_BUNDLE_NAME}; "
            f"found {len(candidates)}."
        )
    bundle_path = candidates[0]
    digest = hashlib.sha256()
    with bundle_path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    INITIAL_EXTRACT_ROOT = PREPARED_ROOT / f"initial_{digest.hexdigest()[:12]}"
    _safe_extract(bundle_path, INITIAL_EXTRACT_ROOT)
    code_archives = sorted(INITIAL_EXTRACT_ROOT.rglob("project_code.zip"))
    if len(code_archives) != 1:
        raise FileNotFoundError("Initial bundle must contain one project_code.zip.")
    PROJECT_ROOT = INITIAL_EXTRACT_ROOT / "project_code"
    _safe_extract(code_archives[0], PROJECT_ROOT)
else:
    configured_project = os.environ.get("RECTANGLE_PROJECT_ROOT")
    if configured_project:
        candidate = Path(configured_project).resolve()
        if (candidate / "data" / "rectangular_dataset.py").is_file():
            PROJECT_ROOT = candidate
    if PROJECT_ROOT is None:
        for candidate in (Path.cwd(), Path.cwd().parent):
            if (candidate / "data" / "rectangular_dataset.py").is_file():
                PROJECT_ROOT = candidate.resolve()
                break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate Phase B project source.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from experiments.run_paths import create_run_directory

configured_data = os.environ.get("RECTANGLE_DATA_INPUT")
if configured_data:
    DATA_ROOT = Path(configured_data).resolve()
    if not _has_initial_data(DATA_ROOT):
        raise FileNotFoundError(
            "RECTANGLE_DATA_INPUT must contain manifest.json and exactly the required "
            "training, validation_calibration, and id_pilot archives."
        )
elif ON_KAGGLE:
    DATA_ROOT = INITIAL_EXTRACT_ROOT / "dataset"
    if not _has_initial_data(DATA_ROOT):
        missing = [name for name in ("manifest.json",) + REQUIRED_INITIAL_ARCHIVES
                   if not (DATA_ROOT / name).is_file()]
        raise FileNotFoundError("Initial data bundle is incomplete; missing: " + ", ".join(missing))
else:
    candidates = (WORK_ROOT / "data" / RUN_ID, PROJECT_ROOT / "data" / RUN_ID)
    DATA_ROOT = next((p.resolve() for p in candidates if _has_initial_data(p)), None)
    if DATA_ROOT is None:
        raise FileNotFoundError(
            "Missing initial Phase B data: manifest.json, training.npz, "
            "validation_calibration.npz, and id_pilot.npz. Generation fallback is disabled."
        )

RESULTS_ROOT = WORK_ROOT / "results" / RUN_ID
RESULT_ROOT = create_run_directory(RESULTS_ROOT)
RUN_VARIANT = RESULT_ROOT.name
OOD_PILOT_RELEASE_APPROVED = False
DEVICE_REQUEST = "cuda"
print(json.dumps({
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "result_root": str(RESULT_ROOT),
    "run_variant": RUN_VARIANT,
    "required_initial_archives": REQUIRED_INITIAL_ARCHIVES,
    "ood_pilot_loaded": False,
    "ood_pilot_release_approved": OOD_PILOT_RELEASE_APPROVED,
    "device_request": DEVICE_REQUEST,
}, indent=2))


## Runtime and initial data

Record runtime details, require CUDA, select cuda:0 explicitly, and check only the three initial archives. Dataset-generation fallback is disabled. OOD pilot is not extracted or loaded here.


In [ ]:
from experiments.rectangle_pipeline import save_environment
import json
import torch

environment = save_environment(RESULT_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is mandatory for Phase B neural-network training. Enable a Kaggle GPU "
        "accelerator and restart the session; CPU fallback is disabled."
    )
if torch.cuda.device_count() < 1:
    raise RuntimeError("CUDA reported available but no CUDA device is visible.")
CUDA_DEVICE_INDEX = 0
torch.cuda.set_device(CUDA_DEVICE_INDEX)
CUDA_DEVICE = torch.device(f"cuda:{CUDA_DEVICE_INDEX}")
DEVICE_REQUEST = "cuda"
if not all(environment.get(key) for key in ("gpu_model", "cuda_version", "pytorch_version")):
    raise RuntimeError("GPU/CUDA/PyTorch environment metadata is incomplete.")
if not environment.get("cuda_available"):
    raise RuntimeError("Saved environment metadata does not confirm CUDA availability.")
for required in ("manifest.json", "training.npz", "validation_calibration.npz", "id_pilot.npz"):
    if not (DATA_ROOT / required).is_file():
        raise FileNotFoundError(f"Required initial input is missing: {DATA_ROOT / required}")
print(json.dumps({
    "python_version": environment["python_version"],
    "pytorch_version": environment["pytorch_version"],
    "cuda_available": environment["cuda_available"],
    "cuda_version": environment["cuda_version"],
    "gpu_model": environment["gpu_model"],
    "gpu_memory_bytes": environment["gpu_memory_bytes"],
    "cpu_model": environment["cpu_model"],
    "logical_cpu_count": environment["logical_cpu_count"],
    "system_memory": environment["system_memory"],
    "selected_device": str(CUDA_DEVICE),
}, indent=2))


In [ ]:
import gc
import json
import torch
from neural_astar.model import NeuralAStarUNet
from ood.feature_extractor import OODFeatureExtractor
from ood.contrastive import nt_xent_loss

if not torch.cuda.is_available():
    raise RuntimeError("CUDA became unavailable before model preflight; training is stopped.")
if DEVICE_REQUEST != "cuda" or CUDA_DEVICE.type != "cuda":
    raise RuntimeError("Preflight requires the explicitly selected CUDA device.")
CUDA_PREFLIGHT_RESULTS = []

def _cuda_step(name, model, inputs, optimizer_factory, expected_shapes, loss_fn):
    model = model.to(CUDA_DEVICE)
    model.train()
    parameters = [p for p in model.parameters() if p.requires_grad]
    if not parameters or any(p.device != CUDA_DEVICE for p in parameters):
        raise RuntimeError(f"{name}: model parameters are not all on CUDA.")
    if any(x.device != CUDA_DEVICE for x in inputs):
        raise RuntimeError(f"{name}: input batches are not on CUDA.")
    optimizer = optimizer_factory(model.parameters())
    optimizer.zero_grad(set_to_none=True)
    first = parameters[0]
    before = first.detach().clone()
    outputs, loss = loss_fn(model, inputs)
    values = outputs if isinstance(outputs, (tuple, list)) else (outputs,)
    shapes = []
    for value in values:
        if not torch.is_tensor(value) or value.device != CUDA_DEVICE:
            raise RuntimeError(f"{name}: forward output is not on CUDA.")
        shapes.append(tuple(int(d) for d in value.shape))
    if tuple(shapes) != tuple(expected_shapes):
        raise RuntimeError(f"{name}: unexpected output shapes {shapes}.")
    if not torch.is_tensor(loss) or loss.device != CUDA_DEVICE or loss.ndim != 0:
        raise RuntimeError(f"{name}: loss is not a scalar CUDA tensor.")
    if not bool(torch.isfinite(loss).item()):
        raise RuntimeError(f"{name}: loss is non-finite.")
    loss.backward()
    gradients = [p.grad for p in parameters if p.grad is not None]
    if not gradients or any(g.device != CUDA_DEVICE for g in gradients):
        raise RuntimeError(f"{name}: gradients are missing or not on CUDA.")
    if any(not bool(torch.isfinite(g).all().item()) for g in gradients):
        raise RuntimeError(f"{name}: non-finite CUDA gradient.")
    optimizer.step()
    torch.cuda.synchronize(CUDA_DEVICE)
    if torch.equal(before, first.detach()):
        raise RuntimeError(f"{name}: CUDA optimizer step did not change a parameter.")
    states = [v for state in optimizer.state.values() for v in state.values()
              if torch.is_tensor(v)]
    if any(value.device != CUDA_DEVICE for value in states):
        raise RuntimeError(f"{name}: optimizer state is not on CUDA.")
    record = {
        "model": name,
        "device": str(CUDA_DEVICE),
        "input_shapes": [tuple(int(d) for d in x.shape) for x in inputs],
        "output_shapes": shapes,
        "parameters_on_cuda": True,
        "forward_backward_optimizer_on_cuda": True,
        "peak_allocated_bytes": int(torch.cuda.max_memory_allocated(CUDA_DEVICE)),
        "peak_reserved_bytes": int(torch.cuda.max_memory_reserved(CUDA_DEVICE)),
    }
    del optimizer, model, inputs, outputs, values, loss, gradients, states, before, first
    gc.collect()
    torch.cuda.empty_cache()
    return record

def _mean_output(model, inputs):
    value = model(inputs[0])
    return value, value.float().mean()

def _contrastive_output(model, inputs):
    first = model.project(model(inputs[0]))
    second = model.project(model(inputs[1]))
    return (first, second), nt_xent_loss(first, second, temperature=0.2)

torch.cuda.reset_peak_memory_stats(CUDA_DEVICE)
x = torch.rand(1, 2, 64, 64, device=CUDA_DEVICE)
CUDA_PREFLIGHT_RESULTS.append(_cuda_step(
    "NeuralAStar", NeuralAStarUNet(), [x],
    lambda p: torch.optim.RMSprop(p, lr=1e-3), ((1, 1, 64, 64),), _mean_output))

torch.cuda.reset_peak_memory_stats(CUDA_DEVICE)
x1 = torch.rand(32, 3, 64, 64, device=CUDA_DEVICE)
x2 = torch.rand(32, 3, 64, 64, device=CUDA_DEVICE)
CUDA_PREFLIGHT_RESULTS.append(_cuda_step(
    "IndependentOODFeatureExtractor", OODFeatureExtractor(), [x1, x2],
    lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=1e-5),
    ((32, 64), (32, 64)), _contrastive_output))

CUDA_PREFLIGHT_COMPLETE = True
print(json.dumps(CUDA_PREFLIGHT_RESULTS, indent=2))


## CUDA preflight and model training

Before training, synthetic inputs verify CUDA model parameters, batches, forward/backward passes, and optimizer updates for the two primary models: Neural A* and the independent OOD encoder. H1, H2, and H3 are optional secondary baselines and are not part of the mandatory preflight.


In [ ]:
from experiments.rectangle_pipeline import train_models

if not CUDA_PREFLIGHT_COMPLETE or DEVICE_REQUEST != "cuda":
    raise RuntimeError("CUDA preflight must pass before model training.")
training_report = train_models(DATA_ROOT, RESULT_ROOT, device="cuda", resume=True)
if training_report.get("device") != "cuda":
    raise RuntimeError("Training did not report CUDA; stop Phase B.")
print(json.dumps(training_report, indent=2))


In [ ]:
from experiments.rectangle_pipeline import calibrate_detector
calibration_report = calibrate_detector(DATA_ROOT, RESULT_ROOT, device="cuda")
print(json.dumps(calibration_report, indent=2))


## Integration checks and pilot

The integration stage uses validation and ID-pilot maps only. OOD-pilot inference is enabled only after model training and D95 calibration are complete and the ID-only integration checks pass. The final holdout and sealed OOD test are not available to these stage functions.

In [ ]:
from experiments.rectangle_pipeline import run_integration_tests
integration_report = run_integration_tests(DATA_ROOT, RESULT_ROOT, device="cuda")
print(json.dumps(integration_report, indent=2))


In [ ]:
import json, shutil, zipfile
from experiments.rectangle_pipeline import cache_pilot_inference

ready_path = RESULT_ROOT / "models" / "model_ready.json"
integration_path = RESULT_ROOT / "integration_tests" / "inference_to_search.json"
if not ready_path.is_file() or not integration_path.is_file():
    raise PermissionError("OOD release requires completed training, D95 calibration, and ID-only integration.")
ready = json.loads(ready_path.read_text(encoding="utf-8"))
integration = json.loads(integration_path.read_text(encoding="utf-8"))
if not (ready.get("training_complete") and ready.get("d95_calibrated")):
    raise PermissionError("OOD pilot stays locked until training and D95 calibration are complete.")
if integration.get("status") != "passed" or integration.get("ood_pilot_loaded"):
    raise PermissionError("OOD pilot stays locked until ID-only integration passes.")
if not OOD_PILOT_RELEASE_APPROVED:
    raise PermissionError("Explicit approval for the separate OOD-pilot upload is required.")

ood_name = "64x64_rectangles_v1_ood_pilot.zip"
ood_bundles = sorted(INPUT_ROOT.rglob(ood_name))
if len(ood_bundles) != 1:
    raise FileNotFoundError(f"Attach exactly one separately approved {ood_name}; found {len(ood_bundles)}.")
release_root = WORK_ROOT / "phase_b_inputs" / "ood_pilot_release"
release_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ood_bundles[0]) as bundle:
    members = set(bundle.namelist())
    required = {"dataset/ood_pilot.npz", "dataset/manifest.json",
                "README_ood_pilot.md", "upload_manifest.json"}
    if members != required:
        raise ValueError("Separate OOD-pilot bundle has an unexpected file list.")
    root = release_root.resolve()
    for member in members:
        target = (release_root / member).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f"Unsafe OOD bundle path: {member}")
    bundle.extractall(release_root)
manifest = json.loads((release_root / "dataset" / "manifest.json").read_text(encoding="utf-8"))
records = manifest.get("maps", [])
if len(records) != 60 or any(row.get("split") != "ood_pilot" for row in records):
    raise ValueError("OOD manifest must contain exactly 60 OOD-pilot records.")
shutil.copyfile(release_root / "dataset" / "ood_pilot.npz", DATA_ROOT / "ood_pilot.npz")
inference_caches = cache_pilot_inference(DATA_ROOT, RESULT_ROOT, device=DEVICE_REQUEST)
print(json.dumps({"ood_pilot_release_gate_passed": True,
                  "ood_pilot_records": len(records),
                  "inference_caches": inference_caches}, indent=2))


In [ ]:
from benchmarks.rectangle_pilot import run_pilot
pilot_report = run_pilot(DATA_ROOT, RESULT_ROOT, device="cuda")
print(json.dumps(pilot_report, indent=2))


## Stop after the pilot

Review the Phase B report and raw pilot outputs. There is no full-benchmark stage. ID holdout and sealed OOD test require separate explicit approval before final evaluation.
